# ODS v2: Motor quotes (MFQ

Builds 8 ods tables in `ods_v2`: `QuoteMotorPrice`, `QuoteMotorPayment`, `QuoteMotorStep`, `QuoteMotorBotSession`, `QuoteMotor`, `QuoteMotorSummary`, `QuoteMotorDriver`, `QuoteMotorVehicle`.

Needs these notebooks to have succeeded first: 20_Quote.

Generated by `ods_build/v2/build_notebook.py` from `entities/*.yaml`: one SQL statement per cell.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ods_build/v2/entities/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# Parameters (set by ODS_v2_Run; leave blank when running by hand)
batch_id   = ""
source_set = ""


In [ ]:
%%pyspark
%run ODS_v2_00_Setup

In [ ]:
%%pyspark
ods_run_log("21_Quote_Motor", "started")


## QuoteMotorPrice


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteMotorPrice                                                              */
/*  Description : Each insurer's cheapest priced row (FinalYearlyPremium > 0) per MFQ quote, with its f */
/*              : ee and scheme.                                                                        */
/*              :                                                                                       */
/*              : Grain: one row per MFQ quote guid and insurer                                         */
/*              : Business key: GuidKey + Insurer                                                       */
/*              : History: current   Cut-off: none                                                      */
/*              : Source: dlk.mfq_quoteresponses + dlk.mfq_quotesummary                                 */
/*              : Reads: dlk.mfq_quoteresponses, dlk.mfq_quotesummary, ods_v2.Quote                     */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteMotorPrice as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  q.QuoteId,
                x.GuidKey,
                x.Insurer,
                x.SchemeName,
                x.FinalYearlyPremium,
                x.Fee,
                x.Premium,
                x.QuoteSummaryId
        from    (
                select  nullif(lower(trim(r.QuoteQueryGuid)), '')                                  as GuidKey,
                        s.Insurer,
                        s.SchemeName,
                        s.FinalYearlyPremium,
                        s.Fee,
                        s.Premium,
                        s.QuoteSummaryId,
                        row_number() over (partition by nullif(lower(trim(r.QuoteQueryGuid)), ''), s.Insurer
                                           order by s.FinalYearlyPremium, s.Fee, s.QuoteSummaryId) as rn
                from    ${ods.dlk}.mfq_quoteresponses   r
                join    ${ods.dlk}.mfq_quotesummary     s   on s.QuoteResponseId = r.QuoteResponseId
                where   s.FinalYearlyPremium > 0
                and     nullif(trim(r.QuoteQueryGuid), '') is not null
                ) x
        join    ${ods.target}.Quote q   on q.SourceSystemId = 2 and q.SourceSystemQuoteReference = x.GuidKey
        where   x.rn = 1
        ) x

## QuoteMotorPayment


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteMotorPayment                                                            */
/*  Description : MFQ online payment attempts tied to the quote (payment -> quote summary -> quote resp */
/*              : onse -> guid; the join RT1                                                            */
/*              : validated on 2026-10-15). PaymentStatus 0 or 5 = successful. Bank and card details ar */
/*              : e not taken.                                                                          */
/*              :                                                                                       */
/*              : Grain: one row per MFQ payment attempt                                                */
/*              : Business key: PaymentId                                                               */
/*              : History: event   Cut-off: none                                                        */
/*              : Source: dlk.mfq_payments + dlk.mfq_quotesummary + dlk.mfq_quoteresponses              */
/*              : Reads: dlk.mfq_payments, dlk.mfq_quotesummary, dlk.mfq_quoteresponses, ods_v2.Quote   */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteMotorPayment as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  q.QuoteId,
                p.ID                                                            as PaymentId,
                nullif(lower(trim(r.QuoteQueryGuid)), '')                                          as GuidKey,
                p.QuoteSummaryId,
                p.QuoteSummaryGuid,
                p.PaymentStatus,
                p.Status,
                case when p.PaymentStatus in (0, 5) then 1 else 0 end           as IsSuccessful,
                p.DateAdded                                                     as PaymentDateTimeUtc,
                p.DateUpdated,
                p.IsFullPayment,
                p.IsDirectDebit,
                p.Amount,
                p.DespositAmount                                                as DepositAmount,
                p.MonthlyAmount,
                p.InsurerID,
                p.PolicyStartDate,
                p.IsKeyCareSelected,
                s.Insurer
        from    ${ods.dlk}.mfq_payments         p
        join    ${ods.dlk}.mfq_quotesummary     s   on s.QuoteSummaryId  = p.QuoteSummaryId
        join    ${ods.dlk}.mfq_quoteresponses   r   on r.QuoteResponseId = s.QuoteResponseId
        join    ${ods.target}.Quote             q   on q.SourceSystemId = 2 and q.SourceSystemQuoteReference = nullif(lower(trim(r.QuoteQueryGuid)), '')
        ) x

## QuoteMotorStep


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteMotorStep                                                               */
/*  Description : Every MFQ journey step a quote passed through, in UTC (dlk.mfq_clientsteps CreatedDat */
/*              : eTime is UTC).                                                                        */
/*              :                                                                                       */
/*              : Grain: one row per MFQ client step event                                              */
/*              : Business key: ClientStepId                                                            */
/*              : History: event   Cut-off: StepDateTimeUtc                                             */
/*              : Source: dlk.mfq_clientsteps                                                           */
/*              : Reads: dlk.mfq_clientsteps, ods_v2.Quote                                              */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteMotorStep as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  s.ClientStepId,
                q.QuoteId,
                nullif(lower(trim(s.QuoteQueryGuid)), '')                                          as GuidKey,
                s.StepId,
                s.CoverCode,
                s.CreatedDateTime                                               as StepDateTimeUtc
        from    ${ods.dlk}.mfq_clientsteps s
        left join ${ods.target}.Quote q on q.SourceSystemId = 2 and q.SourceSystemQuoteReference = nullif(lower(trim(s.QuoteQueryGuid)), '')
        where   s.CreatedDateTime >= date '${ods.cutoff_date}'
        ) x

## QuoteMotorBotSession


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteMotorBotSession                                                         */
/*  Description : Imperva bot classification of MFQ sessions (IsKnownBot, ABPAction), per quote.        */
/*              :                                                                                       */
/*              : Grain: one row per MFQ monitoring session record                                      */
/*              : Business key: MonSessionId                                                            */
/*              : History: event   Cut-off: SessionDateTime                                             */
/*              : Source: dlk.mfq_mon_session                                                           */
/*              : Reads: dlk.mfq_mon_session, ods_v2.Quote                                              */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteMotorBotSession as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  m.Id                                                            as MonSessionId,
                q.QuoteId,
                nullif(lower(trim(m.QuoteId)), '')                                                 as GuidKey,
                m.ImpervaSessionId,
                m.`Timestamp`                                                   as SessionDateTime,
                m.IsKnownBot,
                m.ABPAction
        from    ${ods.dlk}.mfq_mon_session m
        left join ${ods.target}.Quote q on q.SourceSystemId = 2 and q.SourceSystemQuoteReference = nullif(lower(trim(m.QuoteId)), '')
        where   m.`Timestamp` >= date '${ods.cutoff_date}'
        ) x

## QuoteMotor


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_vehicle_make                                                         */
/*  Description : Model code -> make, the most frequent make per code in QuotationStorage (ties alphabe */
/*              : tical).                                                                               */
/*              :                                                                                       */
/*              : Reads: dlk.ext_quotationstorage_riskdetails                                           */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_vehicle_make as
select  code, make
from    (
        select  upper(trim(Car_Model_Code))                         as code,
                upper(trim(Car_Make))                               as make,
                row_number() over (partition by upper(trim(Car_Model_Code))
                                   order by count(*) desc, upper(trim(Car_Make))) as r
        from    ${ods.dlk}.ext_quotationstorage_riskdetails
        where   Car_Model_Code is not null and Car_Make is not null
        group by upper(trim(Car_Model_Code)), upper(trim(Car_Make))
        ) m
where   r = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_mfq_proposer                                                         */
/*  Description : The proposer (PRN 1, highest QuoteDriverId) per quote query, age at the quote's cover */
/*              :  start.                                                                               */
/*              :                                                                                       */
/*              : Reads: dlk.mfq_quotedrivers                                                           */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_mfq_proposer as
select  *
from    (
        select  d.QuoteQueryId, d.DateOfBirth, d.DrivesVehicleUse, d.PenaltyPoints, d.ConvictionsInd,
                d.PenaltyPointsIndicator, d.LicenceType, d.IrelandLicenceDate, d.OccupationCode,
                count(*) over (partition by d.QuoteQueryId)                             as ProposerRows,
                row_number() over (partition by d.QuoteQueryId order by d.QuoteDriverId desc) as r
        from    ${ods.dlk}.mfq_quotedrivers d
        where   d.PRN = 1
        ) p
where   r = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_mfq_driver_summary                                                   */
/*  Description : Driver counts and the youngest date of birth per quote query.                         */
/*              :                                                                                       */
/*              : Reads: dlk.mfq_quotedrivers                                                           */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_mfq_driver_summary as
select  QuoteQueryId,
        count(*)                                                    as DriverCount,
        sum(case when PRN <> 1 then 1 else 0 end)                   as NamedDriverCount,
        max(DateOfBirth)                                            as YoungestDateOfBirth
from    ${ods.dlk}.mfq_quotedrivers
group by QuoteQueryId

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_mfq_step_summary                                                     */
/*  Description : Highest step reached and first / last step time per guid.                             */
/*              :                                                                                       */
/*              : Reads: ods_v2.QuoteMotorStep                                                          */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_mfq_step_summary as
select  GuidKey,
        max(StepId)                                                 as MaxStepId,
        min(case when StepId = 1 then StepDateTimeUtc end)          as FirstStep1DateTimeUtc,
        max(StepDateTimeUtc)                                        as LastStepDateTimeUtc
from    ${ods.target}.QuoteMotorStep
group by GuidKey

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_mfq_price_summary                                                    */
/*  Description : Insurers priced and the cheapest price per guid.                                      */
/*              :                                                                                       */
/*              : Reads: ods_v2.QuoteMotorPrice                                                         */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_mfq_price_summary as
select  GuidKey,
        count(*)                                                    as InsurerCount,
        min(FinalYearlyPremium)                                     as CheapestPremium,
        min_by(Fee, FinalYearlyPremium)                             as CheapestFee,
        min_by(Insurer, FinalYearlyPremium)                         as CheapestInsurer
from    ${ods.target}.QuoteMotorPrice
group by GuidKey

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_mfq_payment_summary                                                  */
/*  Description : Online payment attempted / succeeded per guid.                                        */
/*              :                                                                                       */
/*              : Reads: ods_v2.QuoteMotorPayment                                                       */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_mfq_payment_summary as
select  GuidKey,
        1                                                           as PaymentAttempted,
        max(IsSuccessful)                                           as PaidOnline,
        min(case when IsSuccessful = 1 then PaymentDateTimeUtc end) as FirstSuccessfulPaymentDateTimeUtc
from    ${ods.target}.QuoteMotorPayment
group by GuidKey

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteMotor                                                                   */
/*  Description : One row per MFQ car quote: the latest answers, first / last activity (UTC), reference */
/*              :  and retrieve token, hashed                                                           */
/*              : contact keys (email, mobile, registration, address lines 1+2), vehicle (make from the */
/*              :  QuotationStorage model-code                                                          */
/*              : map), proposer and driver summary, step reached,                                      */
/*              : price and payment summary. As ODS_Build_QuoteJourney QuoteMotor, without the SCV key  */
/*              : (that link is ads).                                                                   */
/*              : Latest query only, by design: MFQ occasionally carries a query / driver record over f */
/*              : rom another quote (known                                                              */
/*              : source bug, not fixed), so earlier versions of a guid are not trusted for attributes  */
/*              : or matching.                                                                          */
/*              : Ages are completed years at cover start; no dates of birth are held. IsGdprWiped = em */
/*              : ail or mobile overwritten                                                             */
/*              : with 'GDPR' (non-consenting, non-buying quotes up to 2026-05-06); the row is kept, co */
/*              : nsumers decide.                                                                       */
/*              :                                                                                       */
/*              : Grain: one row per MFQ quote guid                                                     */
/*              : Business key: GuidKey                                                                 */
/*              : History: current   Cut-off: FirstCreatedDateTimeUtc (through Quote)                   */
/*              : Source: dlk.mfq_quotequery + dlk.mfq_quotedrivers + dlk.mfq_clientsteps + ods QuoteMo */
/*              : torPrice / QuoteMotorPayment                                                          */
/*              : Reads: stg.odsv2_mfq_query_latest, ods_v2.Quote, stg.odsv2_vehicle_make, stg.odsv2_mf */
/*              : q_proposer, stg.odsv2_mfq_driver_summary, stg.odsv2_mfq_step_summary, stg.odsv2_mfq_p */
/*              : rice_summary, stg.odsv2_mfq_payment_summary                                           */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteMotor as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  q.QuoteId,
                l.GuidKey,
                l.QuoteQueryGuid,
                l.QuoteQueryId                                                  as LatestQuoteQueryId,
                l.MaxQuoteQueryId,
                l.QueryRowCount,
                l.FirstCreatedDateTimeUtc,
                l.LastCreatedDateTimeUtc,
                l.ReferenceCode,
                l.AnyReferenceCode,
                l.RetrieveSessionToken,
                l.RetrievalCount,
                l.IsGdprWiped,
                l.EmailKey,
                l.MobileKey,
                l.VehicleRegKey,
                l.EircodeArea,
                l.AddressKey,
                l.CoverStartDate,
                l.PolicyStartDate,
                l.BrokerID,
                case when nullif(trim(l.Gclid), '') is not null then 1 else 0 end as HasGclid,
                l.CoverCode,
                l.CoverVoluntaryExcess,
                l.NCDClaimedYearsEarned,
                l.NCDProtectionRequiredInd,
                l.NCDClaimedBonusProtectionType,
                l.PolicyPreviouslyInsuredInd,
                l.PolicyPreviousInsurer,
                l.HomeownerInd,
                l.VehicleModelCode,
                l.VehicleModelName,
                mk.make                                                         as VehicleMake,
                l.VehicleManufacturedYear,
                l.VehicleFirstRegdYear,
                l.VehicleCubicCapacity,
                l.VehicleImported,
                l.VehicleAnnualMilage,
                l.VehicleValue,
                l.ProposerSex,
                l.ProposerMaritalStatus,
                floor(months_between(l.CoverStartDate, pr.DateOfBirth) / 12)    as ProposerAgeAtCoverStart,
                pr.DrivesVehicleUse                                             as ProposerDrivesVehicleUse,
                pr.PenaltyPoints                                                as ProposerPenaltyPoints,
                pr.PenaltyPointsIndicator                                       as ProposerPenaltyPointsIndicator,
                pr.ConvictionsInd                                               as ProposerConvictionsInd,
                pr.LicenceType                                                  as ProposerLicenceType,
                pr.IrelandLicenceDate                                           as ProposerIrelandLicenceDate,
                pr.OccupationCode                                               as ProposerOccupationCode,
                coalesce(pr.ProposerRows, 0)                                    as ProposerRows,
                coalesce(d.DriverCount, 0)                                      as DriverCount,
                coalesce(d.NamedDriverCount, 0)                                 as NamedDriverCount,
                floor(months_between(l.CoverStartDate, d.YoungestDateOfBirth) / 12) as YoungestDriverAgeAtCoverStart,
                st.MaxStepId,
                st.FirstStep1DateTimeUtc,
                st.LastStepDateTimeUtc,
                coalesce(ps.InsurerCount, 0)                                    as InsurerCount,
                ps.CheapestPremium,
                ps.CheapestFee,
                ps.CheapestInsurer,
                coalesce(py.PaymentAttempted, 0)                                as PaymentAttempted,
                coalesce(py.PaidOnline, 0)                                      as PaidOnline,
                py.FirstSuccessfulPaymentDateTimeUtc,
                l.TestGroupFlag,
                l.VariantFlag,
                l.PricePointType
        from    ${ods.stg}.odsv2_mfq_query_latest       l
        join    ${ods.target}.Quote                     q   on q.SourceSystemId = 2 and q.SourceSystemQuoteReference = l.GuidKey
        left join ${ods.stg}.odsv2_vehicle_make         mk  on mk.code = upper(trim(l.VehicleModelCode))
        left join ${ods.stg}.odsv2_mfq_proposer         pr  on pr.QuoteQueryId = l.MaxQuoteQueryId
        left join ${ods.stg}.odsv2_mfq_driver_summary   d   on d.QuoteQueryId = l.MaxQuoteQueryId
        left join ${ods.stg}.odsv2_mfq_step_summary     st  on st.GuidKey = l.GuidKey
        left join ${ods.stg}.odsv2_mfq_price_summary    ps  on ps.GuidKey = l.GuidKey
        left join ${ods.stg}.odsv2_mfq_payment_summary  py  on py.GuidKey = l.GuidKey
        ) x

## QuoteMotorSummary


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteMotorSummary                                                            */
/*  Description : The warehouse's per-quote MFQ attributes the assets use: first / last device, marketi */
/*              : ng channel, medium and                                                                */
/*              : campaign, quote type and counts, retrievals, sale. Latest EffectiveDate per guid (2,5 */
/*              : 83 guids span two                                                                     */
/*              : month-ends). Rank1Prem* = the cheapest insurer premium by cover (Comp / TPFT) and NCD */
/*              :  protection (NCB = full                                                               */
/*              : protection), as the warehouse holds them. To be derived from dlk with the DWH logic l */
/*              : ater; edw is then                                                                     */
/*              : reconciliation only.                                                                  */
/*              :                                                                                       */
/*              : Grain: one row per MFQ quote guid (latest month-end EffectiveDate)                    */
/*              : Business key: GuidKey                                                                 */
/*              : History: current   Cut-off: CreatedDateTime                                           */
/*              : Source: edw.tbl_fact_mfq                                                              */
/*              : Reads: edw.tbl_fact_mfq, ods_v2.Quote                                                 */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteMotorSummary as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  q.QuoteId,
                m.GuidKey,
                m.EffectiveDate,
                m.CreatedDateTime,
                m.QuoteType,
                m.QQ_All,
                m.FQ_All,
                m.FQ_Orig,
                m.FQ_Retd,
                m.FirstClickFlag,
                m.LastClickFlag,
                m.FirstDevice,
                m.LastDevice,
                m.FirstMktChannel,
                m.LastMktChannel,
                m.FirstMedium,
                m.LastMedium,
                m.FirstCampaign,
                m.LastCampaign,
                m.CampaignName,
                case when nullif(trim(m.GCLID), '') is not null then 1 else 0 end   as HasGclid,
                m.ViewedCoverType,
                m.ViewedBonusProtectionType,
                m.MaxStepId,
                m.MaxStep,
                m.LastStepId,
                m.RetrievalCount,
                nullif(upper(trim(m.RetrievalRef)), '')                                            as RetrievalRef,
                m.ActiveCustomerQuote,
                m.InsurersQuoted,
                m.Rank1PremComp,
                m.Rank1PremTPFT,
                m.Rank1PremCompNCB,
                m.Rank1PremTPFTNCB,
                m.Rank1PremBOTH,
                m.Rank1InsComp,
                m.Rank1InsTPFT,
                m.RiskProfileBand,
                m.QuoteDriverID1,
                m.QuoteDriverID2,
                nullif(upper(trim(m.PolicyCode)), '')                                              as PolicyCode,
                m.ReportingSaleType,
                m.ReportingSaleCategory,
                m.MFQ_Sale,
                m.MFQ_Sale_Type,
                m.SaleMFQ_GWP,
                m.SaleMFQ_Fee,
                m.ImpervaABPaction,
                m.ImpervaKnownBot,
                m.ScraperFlag,
                ods_hash_email(m.CustomerReference)                                 as CustomerReferenceEmailKey
        from    (
                select  *, nullif(lower(trim(QuoteQueryGuid)), '') as GuidKey,
                        row_number() over (partition by nullif(lower(trim(QuoteQueryGuid)), '') order by EffectiveDate desc) as rn
                from    ${ods.edw}.tbl_fact_mfq
                where   nullif(trim(QuoteQueryGuid), '') is not null
                ) m
        left join ${ods.target}.Quote q on q.SourceSystemId = 2 and q.SourceSystemQuoteReference = m.GuidKey
        where   m.rn = 1
        and     m.CreatedDateTime >= date '${ods.cutoff_date}'
        ) x

## QuoteMotorDriver


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_ccm_driver                                                           */
/*  Description : QuotationStorage drivers, latest extract per quotation and PRN.                       */
/*              :                                                                                       */
/*              : Reads: dlk.ext_quotationstorage_driver                                                */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_ccm_driver as
select  *
from    (
        select  d.*, row_number() over (partition by d.Quotation_ID, d.Driver_PRN order by d.`$ExtractDateTime` desc) as rn
        from    ${ods.dlk}.ext_quotationstorage_driver d
        ) x
where   rn = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteMotorDriver                                                             */
/*  Description : Drivers on each motor quote: MFQ (the drivers of the guid's highest QuoteQueryId; whe */
/*              : re MFQ holds two drivers with                                                         */
/*              : the same PRN on a query, the highest QuoteDriverId, as the proposer rule) and Quotati */
/*              : onStorage. Licence                                                                    */
/*              : number is a salted hash (the default 123456 becomes null); names and dates of birth a */
/*              : re not held, the age at                                                               */
/*              : cover start is.                                                                       */
/*              :                                                                                       */
/*              : Grain: one row per quote and driver PRN                                               */
/*              : Business key: SourceSystemId + SourceSystemQuoteReference + DriverPRN                 */
/*              : History: current   Cut-off: none                                                      */
/*              : Source: dlk.mfq_quotedrivers (+ mfq_quotequery), dlk.ext_quotationstorage_driver      */
/*              : Reads: stg.odsv2_mfq_query_latest, dlk.mfq_quotedrivers, ods_v2.Quote, stg.odsv2_ccm_ */
/*              : driver                                                                                */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteMotorDriver as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  q.QuoteId,
                2                                                               as SourceSystemId,
                l.GuidKey                                                       as SourceSystemQuoteReference,
                d.QuoteQueryId,
                d.QuoteDriverId,
                d.PRN                                                           as DriverPRN,
                floor(months_between(l.CoverStartDate, d.DateOfBirth) / 12)     as DriverAgeAtCoverStart,
                d.Sex                                                           as DriverSex,
                ods_hash_id(d.DriverLicenseNumber, '123456')                        as DriverLicenceKey,
                d.LicenceType                                                   as DriverLicenceType,
                d.LicenceCountry                                                as DriverLicenceCountry,
                cast(d.LicenceRestrictionInd as string)                         as DriverLicenceRestrictions,
                d.IrelandLicenceDate                                            as DriverDateLicenceIssued,
                d.RelationshipToProposer                                        as DriverRelationshipToProposer,
                cast(d.DrivesVehicleDrivingFrequency as string)                 as DriverDrivingFrequency,
                cast(d.FullTimeUseOfOtherCar as string)                         as DriverFullTimeUseOfAnotherCar,
                cast(d.PermResident as string)                                  as DriverPermanentResident,
                d.IrelandResidencyDate                                          as DriverResidentSince,
                cast(d.PrevRefusedCover as string)                              as DriverPreviouslyRefusedCover,
                cast(d.NonMotoringConviction as string)                         as DriverNonMotoringConvictions,
                cast(d.PrevRestrictiveTerms as string)                          as DriverPreviouslyImposedTerms,
                cast(d.ProsecutionPending as string)                            as DriverProsecutionPending,
                cast(d.ResidentWithProposer as string)                          as DriverResidentWithProposer,
                cast(d.ClaimsIndicator as string)                               as DriverClaims,
                cast(d.ConvictionsInd as string)                                as DriverConvictions,
                cast(d.MedicalConditionsInd as string)                          as DriverMedicalConditions,
                cast(d.PenaltyPoints as string)                                 as DriverPenaltyPoints,
                d.OccupationCode                                                as DriverOccupation,
                d.OccupationEmployersBusiness                                   as DriverFullTimeEmployersBusiness,
                d.OccupationEmploymentType                                      as DriverEmploymentType
        from    ${ods.stg}.odsv2_mfq_query_latest   l
        join    (
                select  *, row_number() over (partition by QuoteQueryId, PRN order by QuoteDriverId desc) as rn
                from    ${ods.dlk}.mfq_quotedrivers
                ) d     on d.QuoteQueryId = l.MaxQuoteQueryId and d.rn = 1
        join    ${ods.target}.Quote                 q   on q.SourceSystemId = 2 and q.SourceSystemQuoteReference = l.GuidKey
        union all
        select  q.QuoteId,
                9,
                cast(c.Quotation_ID as string),
                null,
                null,
                c.Driver_PRN,
                c.Driver_Age_at_Effective_Date,
                c.Driver_Sex,
                ods_hash_id(c.Driver_Licence_Number, '123456'),
                c.Driver_Licence_Type,
                c.Driver_Country_of_Issue,
                cast(c.Driver_Licence_Restrictions as string),
                c.Driver_Date_Licence_Issued,
                c.Driver_Relationship_to_Proposer,
                cast(c.Driver_Driving_Frequency as string),
                cast(c.Driver_Full_Time_Use_of_Another_Car as string),
                cast(c.Driver_Permanent_Resident as string),
                c.Driver_Resident_Since,
                cast(c.Driver_Previously_Refused_Cover as string),
                cast(c.Driver_Non_Motoring_convictions as string),
                cast(c.Driver_Previously_Imposed_Terms as string),
                cast(c.Driver_Prosecution_Pending as string),
                cast(c.Driver_Resident_with_Proposer as string),
                cast(c.Driver_Claims as string),
                cast(c.Driver_Convictions as string),
                cast(c.Driver_Medical_Conditions as string),
                cast(c.Driver_Penalty_Points as string),
                c.Driver_Occupation,
                c.Driver_Full_Time_Employers_Business,
                c.Driver_Employment_Type
        from    ${ods.stg}.odsv2_ccm_driver c
        join    ${ods.target}.Quote         q   on q.SourceSystemId = 9 and q.SourceSystemQuoteReference = cast(c.Quotation_ID as string)
        ) x

## QuoteMotorVehicle


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.QuoteMotorVehicle                                                            */
/*  Description : The vehicle on each motor quote (MFQ latest answers; QuotationStorage latest extract) */
/*              : . The registration is a                                                               */
/*              : salted hash.                                                                          */
/*              :                                                                                       */
/*              : Grain: one row per motor quote                                                        */
/*              : Business key: SourceSystemId + SourceSystemQuoteReference                             */
/*              : History: current   Cut-off: none                                                      */
/*              : Source: dlk.mfq_quotequery (latest row), dlk.ext_quotationstorage_riskdetails         */
/*              : Reads: stg.odsv2_mfq_query_latest, ods_v2.Quote, stg.odsv2_vehicle_make, dlk.ext_quot */
/*              : ationstorage_riskdetails                                                              */
/******************************************************************************************************/
create or replace table ${ods.target}.QuoteMotorVehicle as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  q.QuoteId,
                2                                                               as SourceSystemId,
                l.GuidKey                                                       as SourceSystemQuoteReference,
                l.VehicleModelCode                                              as CarModelCode,
                mk.make                                                         as CarMake,
                l.VehicleModelName                                              as CarModelName,
                l.VehicleRegKey,
                l.VehicleDateFirstRegistered                                    as DateFirstRegistered,
                l.VehicleDatePurchased                                          as DateFirstPurchased,
                l.VehicleValue,
                cast(l.VehicleBodyType as string)                               as BodyType,
                case when l.VehicleSecurityDeviceInd = true then 'Yes' else 'No' end as VehicleSecurity,
                l.VehicleCubicCapacity                                          as CarCC,
                cast(l.VehicleAreaRating as string)                             as AreaOfMainUse,
                cast(l.VehicleAreaKeptAt as string)                             as AreaKeptOvernight,
                cast(l.VehicleKeptAt as string)                                 as LocationKeptOvernight,
                l.VehiclePleasureMileage                                        as PrivateMileage,
                l.VehicleBusinessMileage                                        as BusinessMileage
        from    ${ods.stg}.odsv2_mfq_query_latest   l
        join    ${ods.target}.Quote                 q   on q.SourceSystemId = 2 and q.SourceSystemQuoteReference = l.GuidKey
        left join ${ods.stg}.odsv2_vehicle_make     mk  on mk.code = upper(trim(l.VehicleModelCode))
        union all
        select  q.QuoteId,
                9,
                cast(r.Quotation_ID as string),
                r.Car_Model_Code,
                r.Car_Make,
                r.Car_Model_Name,
                ods_hash_id(r.Registration_Number, ''),
                r.Date_First_Registered,
                r.Date_First_Purchased,
                r.Value,
                cast(r.Body_Type as string),
                cast(r.Vehicle_Security as string),
                r.Car_CC,
                cast(r.Area_of_Main_Use as string),
                cast(r.Area_Kept_Overnight as string),
                cast(r.Location_Kept_Overnight as string),
                r.Private_Mileage,
                r.Business_Mileage
        from    (
                select  *, row_number() over (partition by Quotation_ID order by `$ExtractDateTime` desc) as rn
                from    ${ods.dlk}.ext_quotationstorage_riskdetails
                ) r
        join    ${ods.target}.Quote q   on q.SourceSystemId = 9 and q.SourceSystemQuoteReference = cast(r.Quotation_ID as string)
        where   r.rn = 1
        ) x

## Checks


In [ ]:
%%pyspark
# Checks for the tables this notebook built. All are checked and logged to OdsBuildLog; the cell then fails if
# any check failed, naming the table and the likely cause.
import json, re
from functools import reduce
from pyspark.sql import functions as F
SPEC = json.loads(r"""[{"table": "QuoteMotorPrice", "bk": ["GuidKey", "Insurer"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteMotorPayment", "bk": ["PaymentId"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteMotorStep", "bk": ["ClientStepId"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteMotorBotSession", "bk": ["MonSessionId"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteMotor", "bk": ["GuidKey"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteMotorSummary", "bk": ["GuidKey"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteMotorDriver", "bk": ["SourceSystemId", "SourceSystemQuoteReference", "DriverPRN"], "sk": null, "nullable": [], "allow": []}, {"table": "QuoteMotorVehicle", "bk": ["SourceSystemId", "SourceSystemQuoteReference"], "sk": null, "nullable": [], "allow": []}]""")
PII_NAME = re.compile(r"(e_?mail|forename|first_?name|surname|last_?name|full_?name|second_?name|phone|mobile|telephone|post_?code|eircode(?!_?area)|address_?line|risk_?address|postal_?address|dob$|date_?of_?birth|licen[cs]e_?num|registration_?no|reg_?number|registration_number|^ani$|^dnis$|originatingdnis|^payer_|iban|^bic|^contact$|rendered_?message)", re.I)
EMAIL_RE = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
MOBILE_RE = r"(^|[^0-9A-Za-z-])(\+?353 ?|0)8[35-9][ -]?[0-9]{3}[ -]?[0-9]{4}([^0-9A-Za-z-]|$)"   # stands alone, not inside an id
rows, failures = [], []
for s in SPEC:
    t = f"{target_schema}.{s['table']}"
    df = spark.table(t)
    bk = s["bk"]
    must = [c for c in bk if c not in s["nullable"]]
    agg = df.agg(F.count(F.lit(1)).alias("n"),
                 F.countDistinct(F.struct(*bk)).alias("d_bk"),
                 F.sum(F.when(reduce(lambda a, b: a | b, [F.col(c).isNull() for c in must] or [F.lit(False)]), 1).otherwise(0)).alias("null_bk"),
                 *( [F.countDistinct(s["sk"]).alias("d_sk")] if s["sk"] else [] )).first()
    pii_cols = [c for c in df.columns if PII_NAME.search(c) and not c.endswith("Key") and c not in s["allow"]]
    text_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]   # hashes never match
    hits = {}
    if text_cols:
        found = df.select(*text_cols).limit(1000000).agg(
            *[F.sum((F.col(c).rlike(EMAIL_RE) | F.col(c).rlike(MOBILE_RE)).cast("int")).alias(c) for c in text_cols]).first().asDict()
        hits = {c: v for c, v in found.items() if v}
    problems, advice = [], []
    keys = " + ".join(bk)
    if agg["n"] == 0:
        problems.append("empty")
        advice.append("the table is empty: its source is empty, or a filter (cut-off date, join) removed every row")
    if agg["d_bk"] != agg["n"]:
        problems.append(f"{agg['n'] - agg['d_bk']:,} duplicate business keys")
        advice.append(f"several rows share a key ({keys}): the source sent the same record more than once, or the "
                      f"dedupe rule in the cell no longer holds. Examples: select {', '.join(bk)}, count(*) from {t} "
                      f"group by {', '.join(bk)} having count(*) > 1")
    if agg["null_bk"]:
        problems.append(f"{agg['null_bk']:,} null business keys")
        advice.append(f"key columns ({', '.join(must)}) are empty on some rows: if that is legitimate for a column, add it "
                      f"to key_nullable for this table in the config; otherwise the source changed")
    if s["sk"] and agg["d_sk"] != agg["n"]:
        problems.append("surrogate key not unique")
        advice.append("the surrogate key repeats: the business key is not unique (see above)")
    if pii_cols:
        problems.append(f"PII-named columns {pii_cols}")
        advice.append("columns named like personal data: hash them (HASH_*) or leave them out in the config")
    if hits:
        problems.append(f"email/mobile values in {hits}")
        advice.append("emails / mobile numbers inside these columns: wrap the column in REDACT() (or hash it) in the config")
    rows.append((batch_id, s["table"], agg["n"], agg["n"] - agg["d_bk"], agg["null_bk"], "; ".join(problems) or "ok"))
    print(f"{s['table']:<28} {agg['n']:>13,}  {'; '.join(problems) or 'ok'}")
    if problems:
        failures.append(f"{s['table']}: " + " | ".join(advice))
log = spark.createDataFrame(rows, "OdsBatchId string, TableName string, RowCount long, DuplicateKeys long, NullKeys long, Result string")
log.withColumn("CheckedAt", F.current_timestamp()).write.mode("append").saveAsTable(f"{target_schema}.OdsBuildLog")
if failures:
    ods_run_log("21_Quote_Motor", "failed", "; ".join(f.split(":")[0] for f in failures))
    raise AssertionError("ods checks failed:\n- " + "\n- ".join(failures))
ods_run_log("21_Quote_Motor", "succeeded")
